In [15]:
# ==========================================
# 品番ごとTXT + 対応表CSV → BERT埋め込み比較
# ==========================================
import os, glob, re
import numpy as np
import pandas as pd

# ★★ ここを環境に合わせて設定 ★★
REVIEWS_DIR = "../data/reviews"  # 例: /mnt/data/reviews に  A001.txt, MG125201.txt ... が入っている
META_CSV    = "../data/csv/フリーワード分析テキスト対応表.csv"  # 例: 品番,売上規模

# 依存の確認（未インストールなら: pip install -U sentence-transformers）
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine

def read_txt_as_single_doc(path: str) -> str:
    # 改行区切りレビューを1文書にまとめる（空行除去・スペース正規化）
    with open(path, encoding="utf-8") as f:
        txt = f.read()
    lines = [ln.strip() for ln in txt.splitlines() if ln.strip()]
    doc = "。".join(lines)
    # 連続空白を1つに
    doc = re.sub(r"\s+", " ", doc)
    return doc

# -------------------------
# 1) 対応表読み込み
# -------------------------
meta = pd.read_csv(META_CSV)
if not set(["品番","売上規模"]).issubset(meta.columns):
    raise ValueError("対応表CSVに「品番」「売上規模」列が必要です。列名を確認してください。")

# 売上規模は「大」/ それ以外（小 など）で2群比較
meta = meta.astype({"品番": str, "売上規模": str})

# -------------------------
# 2) レビューTXTを集約
# -------------------------
records = []
txt_files = sorted(glob.glob(os.path.join(REVIEWS_DIR, "*.txt")))
if not txt_files:
    raise FileNotFoundError(f"{REVIEWS_DIR} に .txt が見つかりません。パスを確認してください。")

for fp in txt_files:
    pid = os.path.splitext(os.path.basename(fp))[0]  # 例: MG125201
    doc = read_txt_as_single_doc(fp)

    # 対応表から売上規模を取得（見つからなければ「不明」）
    row = meta.loc[meta["品番"] == pid]
    scale = row["売上規模"].iloc[0] if len(row) else "不明"

    records.append({"品番": pid, "売上規模": scale, "レビュー": doc})

df = pd.DataFrame(records).sort_values("品番").reset_index(drop=True)

print("✅ 読み込み結果（先頭）：")
print(df.head())
missing = (df["売上規模"] == "不明").sum()
if missing:
    print(f"⚠️ 対応表に存在しない品番が {missing} 件あります（「不明」扱い）。CSVの品番とTXTファイル名の一致を確認してください。")

# -------------------------
# 3) BERT埋め込み
# -------------------------
# 日本語Sentence-BERT（平均プーリング済み）モデル
model = SentenceTransformer("sonoisa/sentence-bert-base-ja-mean-tokens")
emb = model.encode(df["レビュー"].tolist(), convert_to_tensor=False)  # shape: [N, D]
emb = np.asarray(emb)

# -------------------------
# 4) 2群（large=1,2 vs small=0）の平均ベクトル距離
# -------------------------
df["売上規模"] = pd.to_numeric(df["売上規模"], errors="coerce")
mask_large = df["売上規模"].isin([1, 2])
mask_small = (df["売上規模"] == 0) & (df["売上規模"] != "不明")

if mask_large.sum() == 0 or mask_small.sum() == 0:
    print("⚠️ large群またはsmall群のデータが不足しています。対応表を確認してください。")
else:
    mean_large = emb[mask_large].mean(axis=0)
    mean_small = emb[mask_small].mean(axis=0)
    dist = cosine(mean_large, mean_small)
    print(f"\n📐 平均ベクトル間コサイン距離（large[1,2] vs small[0]）: {dist:.4f}")

# -------------------------
# 5) 参考：結果を保存（任意）
# -------------------------
# 各品番の埋め込みをCSVに落とす（次の分析に便利）
EMB_CSV = "../data/csv/review_embeddings.csv"
emb_df = pd.DataFrame(emb)
out = pd.concat([df[["品番","売上規模"]].reset_index(drop=True), emb_df], axis=1)
out.to_csv(EMB_CSV, index=False, encoding="utf-8")
print(f"💾 埋め込みCSVを保存: {EMB_CSV}")

# -------------------------
# 6) （任意）上位/下位の近傾向サンプル確認
# -------------------------
# 品番ごとの「大」平均に対するコサイン距離→小さいほど『大』寄りの言語傾向
if mask_large.sum() and mask_small.sum():
    from numpy.linalg import norm
    def cos_dist(a,b): return 1 - np.dot(a,b)/(norm(a)*norm(b) + 1e-12)
    dists_to_large_mean = [cos_dist(v, mean_large) for v in emb]
    df["大平均からの距離"] = dists_to_large_mean
    top_like_large = df.sort_values("大平均からの距離").head(5)[["品番","売上規模","大平均からの距離"]]
    print("\n🔎 『大』平均に言語的に近い上位サンプル（距離が小）:")
    print(top_like_large.to_string(index=False))

No sentence-transformers model found with name sonoisa/sentence-bert-base-ja-mean-tokens. Creating a new one with mean pooling.


✅ 読み込み結果（先頭）：
         品番 売上規模                                               レビュー
0  ME118307    2  着け心地が最高。ワイヤーなしでバストアップが不安でしたが、。しっかりと胸を包んでバストしてく...
1  ME124302    0  フロントホックシリーズはかなりお気に入りです！しっかりホールド感があり、ホックも留めやすいの...
2  ME125102    0  一目惚れしました。色もデザインも好み過ぎて最高です！。着心地。締め付けはないけど、それなりの...
3  ME125301    1  旅行先で替えの下着を忘れたことに気づき、急遽入った百貨店内の店舗で、色々と試着させていただい...
4  ME125302    0  以前からブラデリスのナイトブラ使用中です。いつもオールインワンブラをよく購入するのですが今回...

📐 平均ベクトル間コサイン距離（large[1,2] vs small[0]）: 0.0232
💾 埋め込みCSVを保存: ../data/csv/review_embeddings.csv

🔎 『大』平均に言語的に近い上位サンプル（距離が小）:
      品番  売上規模  大平均からの距離
ME118307     2  0.022887
MG122201     2  0.026730
ME125102     0  0.036015
ME125301     1  0.042489
ME524302     0  0.043986
